# 🗣️ 拾語者 Speak My Life — 雲端算力（Colab GPU）

把 **語音合成（GPT-SoVITS）** 的算力搬到 Colab 免費 GPU；可自由加開 **生圖（SD-Turbo）** 與 **文字／復健評分（Qwen）**。手機 App 與網頁版都能連上，由雲端代跑。

> **為什麼不用 Google Apps Script？** 它是 Workspace 的 JavaScript 自動化，**沒有 GPU、不能跑 Python/PyTorch**，無法執行這些模型。GPU 推論只能用 Colab。

### 運作方式
```
  [手機 App]─┐                                   ┌ api_v2 :9880  語音 GPT-SoVITS
           ├ https 公開網址(Cloudflare) ─ 橋接:9879 ┼ :9881         生圖 SD-Turbo（選用）
  [網頁版] ─┘   app 走 /tts /models；web 走 /speak /voices /image /text（含 CORS）  └ :9882  文字 Qwen（選用）
```

### 事前準備（一次）
1. **語音模型上傳 Google Drive**：`我的雲端硬碟/Speak-My-Life/voice_models/`（＝手機 `/sdcard/Speak-My-Life/voice_models/` 那整包）。更名前建的 `VoiceWeaver/` 會在執行時自動改名成 `Speak-My-Life/`。只跑生圖/文字、不跑語音則可略過。
2. Colab 上方 **執行階段 → 變更執行階段類型 → T4 GPU**。

> ⚠️ 不含任何語音模型；由你自行上傳有權使用的素材。GPT-SoVITS 為 AGPL-3.0。


## ① 自由選擇：這台 Colab 要跑哪些功能


In [ ]:
# 想跑哪個就設 True（語音最吃準備、生圖/文字可獨立開）。T4 記憶體有限，三個全開可能吃緊。
RUN_VOICE = True    # GPT-SoVITS 語音合成（:9880）
RUN_IMAGE = False   # SD-Turbo 生圖（:9881）
RUN_TEXT  = False   # Qwen 文字推論／復健評分（:9882）
assert RUN_VOICE or RUN_IMAGE or RUN_TEXT, '至少要開一個功能'
print('將啟動：', ' '.join(k for k,v in [('語音',RUN_VOICE),('生圖',RUN_IMAGE),('文字',RUN_TEXT)] if v))


## ② 確認 GPU


In [ ]:
import subprocess
r = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(r.stdout.split(chr(10))[8] if r.returncode==0 and len(r.stdout.split(chr(10)))>8 else r.stdout or r.stderr)
assert r.returncode==0, '❌ 沒有 GPU！執行階段 → 變更執行階段類型 → GPU (T4)，再重跑。'
print('✅ GPU 就緒')


## ③ 安裝相依（約 3–6 分鐘）


In [ ]:
import os
ROOT='/content/GPT-SoVITS'
if RUN_VOICE and not os.path.exists(ROOT):
    !git clone --depth 1 https://github.com/RVC-Boss/GPT-SoVITS $ROOT
    # Colab 修正：requirements.txt 開頭的 `--no-binary=opencc` 會強制 opencc 從源碼編譯 → 在 Colab 失敗
    # （Failed building wheel for opencc）。移除該指令，改用 PyPI 預編譯 wheel（Linux 有 wheel、免編譯）。
    !sed -i '/--no-binary=opencc/d' $ROOT/requirements.txt
    !cd $ROOT && pip -q install -r requirements.txt 2>&1 | tail -3
    # 保險：萬一 opencc 仍裝不起來，退用純 Python 版（相同 `import opencc` / OpenCC API，免編譯）
    !python -c "import opencc" 2>/dev/null || pip -q install opencc-python-reimplemented
if RUN_IMAGE:
    !pip -q install diffusers transformers accelerate 2>&1 | tail -2
if RUN_TEXT:
    !pip -q install 'transformers>=4.40' accelerate 2>&1 | tail -2
!pip -q install 'huggingface_hub>=0.23'
print('✅ 安裝完成')

## ④ 語音模型（RUN_VOICE 才需要）— 掛 Drive，情緒資料夾結構

Drive 結構（依語言分，每角色一組情緒參考音）：
```
MyDrive/Speak-My-Life/      （更名前的 MyDrive/VoiceWeaver/ 會自動改名過來）
  ZH/ 遏蝶/  ├ 遏蝶-e10.ckpt  ├ 遏蝶_e10_s450.pth
            └ reference_audios/**/emotions/【情緒】逐字稿.wav
  EN/ …  JA/ …  KO/ …
```
檔名的【情緒】＋逐字稿會被自動解析；合成時依情緒挑對應參考音（換情緒＝換語氣）。


In [ ]:
import os, re, json, time
MODELS_ROOT='/content'   # 不跑語音時的佔位
if RUN_VOICE:
    from huggingface_hub import snapshot_download
    snapshot_download('lj1995/GPT-SoVITS', local_dir=f'{ROOT}/GPT_SoVITS/pretrained_models', local_dir_use_symlinks=False)
    # ── 合成缺件補齊（與桌面端同組修復；缺這些會「連得上卻無法合成」）──
    _PM=f'{ROOT}/GPT_SoVITS/pretrained_models'
    os.makedirs(f'{_PM}/fast_langdetect', exist_ok=True)   # langsegmenter 指到此目錄，不存在直接在 /tts 報錯
    if not os.path.isfile(f'{_PM}/sv/pretrained_eres2netv2w24s4ep4.ckpt'):  # ERes2Net：載 SoVITS 權重時要用，缺→/set_sovits_weights 回 400
        try:
            from huggingface_hub import hf_hub_download
            hf_hub_download('lj1995/GPT-SoVITS','sv/pretrained_eres2netv2w24s4ep4.ckpt',local_dir=_PM,local_dir_use_symlinks=False)
        except Exception as _e:
            print('⚠️ sv/ERes2Net 補下載失敗（v2Pro/v4 角色可能無法合成）：',_e)
    from google.colab import drive; drive.mount('/content/drive')
    # 更名搬遷（與 Android／網頁／電腦同一條規則）：先找 Speak-My-Life；沒有但有舊的
    # VoiceWeaver → os.rename 改名（掛載的 Drive 上等同改名）；改名失敗就沿用舊資料夾。
    _MYDRIVE='/content/drive/MyDrive'
    _NEW, _OLD = f'{_MYDRIVE}/Speak-My-Life', f'{_MYDRIVE}/VoiceWeaver'
    MODELS_ROOT=_NEW
    if not os.path.exists(_NEW) and os.path.isdir(_OLD):
        try:
            os.rename(_OLD, _NEW); print('📁 已把 Drive 的 VoiceWeaver/ 改名為 Speak-My-Life/')
        except Exception as _e:
            MODELS_ROOT=_OLD; print('⚠️ Drive 的 VoiceWeaver/ 改名失敗，沿用舊資料夾：', _e)

    # 沒有就建好 Speak-My-Life/ 與 ZH/EN/JA/KO 範本資料夾（不再 hard-fail，直接引導）
    made=[]
    for d in [MODELS_ROOT]+[f'{MODELS_ROOT}/{l}' for l in ('ZH','EN','JA','KO')]:
        if not os.path.isdir(d):
            os.makedirs(d, exist_ok=True); made.append(os.path.basename(d))
    if made:
        print(f'📁 已建立資料夾範本：{made}（把「角色資料夾」放進對應語言夾即可）')

    # 【情緒】逐字稿.wav → (canon情緒, 逐字稿)；沒【】→ (中立, 檔名)。與 Mac 端 emotion_models 同一套。
    _ALIAS={'中立':['中立','平静','平靜','neutral'],'开心':['开心','開心','高兴','高興','快乐','快樂','喜悦','happy'],
            '难过':['难过','難過','伤心','傷心','难受','難受','sad'],'恐惧':['恐惧','恐懼','害怕','惊恐','驚恐','fear'],
            '吃惊':['吃惊','吃驚','惊讶','驚訝','惊喜','驚喜','surprise']}
    _CANON={a:c for c,al in _ALIAS.items() for a in al}
    def _parse(fn):
        stem=os.path.splitext(fn)[0]
        m=re.match(r'^\s*[【\[]\s*([^】\]]+?)\s*[】\]]\s*(.*)$', stem)
        if m: return _CANON.get(m.group(1).strip(), m.group(1).strip() or '中立'), m.group(2).strip()
        return '中立', stem.strip()
    def _sz(p):
        try: return os.path.getsize(p)
        except OSError: return 0

    # ── 掃曲庫 → 逐角色健檢報告 + 雲端曲庫目錄索引 catalog.json ──
    cat={'version':1,'updatedAt':int(time.time()*1000),'characters':[],
         'summary':{'characters':0,'ready':0,'bytes':0}}
    ready=total=0
    for lang in sorted(os.listdir(MODELS_ROOT)):
        ldir=f'{MODELS_ROOT}/{lang}'
        if lang.upper() not in ('ZH','EN','JA','KO') or not os.path.isdir(ldir):
            continue
        LANG=lang.upper()
        for char in sorted(os.listdir(ldir)):
            cdir=f'{ldir}/{char}'
            if not os.path.isdir(cdir):
                continue
            total+=1
            gpt=sov=None
            for f in os.listdir(cdir):
                ap=f'{cdir}/{f}'
                if f.lower().endswith('.ckpt'): gpt={'path':os.path.relpath(ap,MODELS_ROOT),'size':_sz(ap)}
                elif f.lower().endswith('.pth'): sov={'path':os.path.relpath(ap,MODELS_ROOT),'size':_sz(ap)}
            refs=[]; emos=[]; untag=[]
            for r,_d,fs in os.walk(f'{cdir}/reference_audios'):
                for f in sorted(fs):
                    if not f.lower().endswith(('.wav','.mp3','.flac')):
                        continue
                    ap=os.path.join(r,f); emo,prompt=_parse(f)
                    if emo not in emos: emos.append(emo)
                    if not re.match(r'^\s*[【\[]', os.path.splitext(f)[0]): untag.append(f)
                    refs.append({'path':os.path.relpath(ap,MODELS_ROOT),'emotion':emo,'prompt':prompt,'size':_sz(ap)})
            ok=bool(gpt and sov and refs); ready+=1 if ok else 0
            b=(gpt or {}).get('size',0)+(sov or {}).get('size',0)+sum(x['size'] for x in refs)
            cat['characters'].append({'name':f'{char} {LANG}','character':char,'lang':LANG,'ok':ok,
                'emotions':sorted(emos),'gpt':gpt,'sovits':sov,'refs':refs,'bytes':b})
            cat['summary']['characters']+=1; cat['summary']['ready']+=1 if ok else 0; cat['summary']['bytes']+=b
            issues=[]
            if not gpt: issues.append('缺 *.ckpt')
            if not sov: issues.append('缺 *.pth')
            if not refs: issues.append('reference_audios 沒有參考音')
            elif untag: issues.append(f'{len(untag)} 個檔名沒【情緒】前綴→當中立')
            tail=('  ['+'; '.join(issues)+']') if issues else ''
            print(('✅' if ok else '⚠️')+f' {LANG}/{char}  情緒:{"／".join(sorted(emos)) or "（無）"}{tail}')

    # 寫出目錄索引（Mac 端 emotion_models.write_catalog 同格式；隨選下載/上傳/手機瀏覽都靠它）
    try:
        with open(f'{MODELS_ROOT}/catalog.json','w',encoding='utf-8') as fp:
            json.dump(cat, fp, ensure_ascii=False, indent=1)
        print(f"🗂 已更新曲庫目錄 catalog.json：{total} 角色 / 約 {cat['summary']['bytes']/(1024*1024):.0f} MB")
    except Exception as e:
        print(f'⚠️ 目錄索引寫入失敗：{e}')
    print(f'✅ 共 {total} 個角色，可用 {ready}（權重由 Drive 直接載入，切換角色時才讀）')
    if total==0:
        print('（還沒有任何角色。結構：<語言>/<角色>/{*.ckpt,*.pth, reference_audios/**/emotions/【情緒】逐字稿.wav}）')
else:
    print('（跳過語音）')

## ⑤ 啟動所選服務


In [ ]:
import base64, subprocess, os, time, urllib.request
def _wait(url,label,secs=150):
    for _ in range(secs//3):
        time.sleep(3)
        try:
            with urllib.request.urlopen(url,timeout=3) as r:
                if r.status==200: print(f'✅ {label} 就緒'); return True
        except Exception: pass
    print(f'⚠️ {label} 逾時'); return False
procs={}
if RUN_VOICE:
    os.chdir(ROOT)
    _venv=dict(os.environ)
    try:
        import certifi; _venv.setdefault('SSL_CERT_FILE',certifi.where()); _venv.setdefault('REQUESTS_CA_BUNDLE',certifi.where())
    except Exception: pass
    procs['voice']=subprocess.Popen(['python','api_v2.py','-a','127.0.0.1','-p','9880'], env=_venv,
        stdout=open('/content/voice.log','w'),stderr=subprocess.STDOUT)
    _wait('http://127.0.0.1:9880/health','語音 api_v2')
if RUN_IMAGE:
    open('/content/img.py','wb').write(base64.b64decode('IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiLmi77oqp7ogIUgU3BlYWsgTXkgTGlmZSDmnKzmqZ/nlJ/lnJbkvLrmnI3lmajvvIhNYWMg56uv77yM5YWN6YeR6ZGw77yJ44CCCgrot58gR1BULVNvVklUUyDkuIDmqKPlnKjjgIzpm7vohabmnKzmqZ/jgI3pgYvnrpfvvJrmiYvmqZ/pgI/pgY4gVVNCKGFkYiByZXZlcnNlKSDmiJYgV2ktRmkg6YCj5YiwIDo5ODgx77yMCuS4jemcgOimgeS7u+S9lembsuerryBBUEkg6YeR6ZGw44CC55SoIGRpZmZ1c2VycyArIEFwcGxlIE1QUyDot5EgU0QtVHVyYm/vvIjlv6vjgIE1MTJweOOAgemBqeWQiCBBQUMg5ZyW5Y2h77yJ44CCCgrnq6/pu57vvJoKICAgIEdFVCAgL2hlYWx0aCAgICAgICAgICAgICAtPiB7b2ssIHJlYWR5LCBtb2RlbCwgZGV2aWNlLCBlcnJvcn0KICAgIFBPU1QgL2dlbmVyYXRlICAgICAgICAgICBib2R5IHsicHJvbXB0IjogIi4uLiIsICJzdGVwcyI6IDJ9ICAtPiBpbWFnZS9wbmcKCuaooeWei+WPr+eUqOeSsOWig+iuiuaVuOimhuiTi++8mgogICAgSU1BR0VfR0VOX01PREVM77yI6aCQ6KitIHN0YWJpbGl0eWFpL3NkLXR1cmJv77yb5oOz5pu057K+57e75Y+v55SoIHN0YWJpbGl0eWFpL3NkeGwtdHVyYm/vvIkKICAgIElNQUdFX0dFTl9TVEVQU++8iOmgkOiorSAy77yJCiAgICBTTUxfSU1BR0VfUE9SVO+8iOmgkOiorSA5ODgx77yJCiIiIgpmcm9tIF9fZnV0dXJlX18gaW1wb3J0IGFubm90YXRpb25zCgppbXBvcnQgaW8KaW1wb3J0IGpzb24KaW1wb3J0IG9zCmltcG9ydCByZQppbXBvcnQgc29ja2V0CmltcG9ydCBzdWJwcm9jZXNzCmltcG9ydCBzeXMKaW1wb3J0IHRocmVhZGluZwppbXBvcnQgdGltZQpmcm9tIGh0dHAuc2VydmVyIGltcG9ydCBCYXNlSFRUUFJlcXVlc3RIYW5kbGVyLCBUaHJlYWRpbmdIVFRQU2VydmVyCgoKZGVmIHNlbGZfaW5mbygpIC0+IGRpY3Q6CiAgICAiIiLlm57loLHmnKzmqZ/kuLvmqZ/lkI3oiIcgVGFpbHNjYWxlIElQ77yIMTAwLjY0LjAuMC8xMO+8ie+8jOe1puaJi+apn+iHquWLleWKoOWFpembu+iFpua4heWWrueUqOOAgiIiIgogICAgaG9zdCA9IHNvY2tldC5nZXRob3N0bmFtZSgpLnNwbGl0KCIuIilbMF0gb3IgIumbu+iFpiIKICAgIHRzaXAgPSAiIgogICAgdHJ5OgogICAgICAgIG91dCA9IHN1YnByb2Nlc3MucnVuKFsiaWZjb25maWciXSwgY2FwdHVyZV9vdXRwdXQ9VHJ1ZSwgdGV4dD1UcnVlLCB0aW1lb3V0PTQpLnN0ZG91dAogICAgICAgIGZvciBtIGluIHJlLmZpbmRpdGVyKHIiaW5ldCAoMTAwXC5cZCtcLlxkK1wuXGQrKSIsIG91dCk6CiAgICAgICAgICAgIGlwID0gbS5ncm91cCgxKQogICAgICAgICAgICBpZiA2NCA8PSBpbnQoaXAuc3BsaXQoIi4iKVsxXSkgPD0gMTI3OiAgICMgMTAwLjY0LjAuMC8xMAogICAgICAgICAgICAgICAgdHNpcCA9IGlwCiAgICAgICAgICAgICAgICBicmVhawogICAgZXhjZXB0IEV4Y2VwdGlvbjoKICAgICAgICBwYXNzCiAgICByZXR1cm4geyJvayI6IFRydWUsICJob3N0bmFtZSI6IGhvc3QsICJ0YWlsc2NhbGVfaXAiOiB0c2lwfQoKUE9SVCA9IGludChvcy5lbnZpcm9uLmdldCgiU01MX0lNQUdFX1BPUlQiLCAiOTg4MSIpKQpNT0RFTCA9IG9zLmVudmlyb24uZ2V0KCJJTUFHRV9HRU5fTU9ERUwiLCAic3RhYmlsaXR5YWkvc2QtdHVyYm8iKQpERUZBVUxUX1NURVBTID0gaW50KG9zLmVudmlyb24uZ2V0KCJJTUFHRV9HRU5fU1RFUFMiLCAiMiIpKQoKX3BpcGUgPSBOb25lCl9sb2NrID0gdGhyZWFkaW5nLkxvY2soKSAgICAgICAjIOWWruW8teW6j+WIl+WMlu+8jOmBv+WFjSBNUFMg6KiY5oa26auU54iG5o6JCl9sb2FkX2VyciA9ICIiCl9kZXZpY2UgPSAiY3B1IgpfYnVzeSA9IDAgICAgICAgICAgICAgICAgICAgICAgIyDpgLLooYzkuK3nmoTnlJ/lnJbmlbjvvIjntabmjqfliLbkuK3lv4Ppoa/npLrjgIzlkIjmiJDpgLLluqbjgI3vvIkKX2RvbmUgPSAwICAgICAgICAgICAgICAgICAgICAgIyDntK/oqIjlrozmiJDlvLXmlbgKX2xhc3RfcHJvbXB0ID0gIiIKCgpkZWYgbG9hZF9waXBlKCkgLT4gTm9uZToKICAgICIiIuiDjOaZr+i8ieWFpeaooeWei++8iOmmluasoeacg+iHquWLleS4i+i8ie+8ie+8jOiukyBzZXJ2ZXIg6IO956uL5Yi75Zue5oeJIC9oZWFsdGjjgIIiIiIKICAgIGdsb2JhbCBfcGlwZSwgX2xvYWRfZXJyLCBfZGV2aWNlCiAgICB0cnk6CiAgICAgICAgaW1wb3J0IHRvcmNoCiAgICAgICAgZnJvbSBkaWZmdXNlcnMgaW1wb3J0IEF1dG9QaXBlbGluZUZvclRleHQySW1hZ2UKCiAgICAgICAgIyDoo53nva7lhKrlhYjluo/vvJpDVURB77yIV2luZG93cy9MaW51eCBOVklESUHvvInihpIgTVBT77yIQXBwbGXvvInihpIgQ1BV44CCCiAgICAgICAgaWYgdG9yY2guY3VkYS5pc19hdmFpbGFibGUoKToKICAgICAgICAgICAgX2RldmljZSA9ICJjdWRhIgogICAgICAgIGVsaWYgZ2V0YXR0cih0b3JjaC5iYWNrZW5kcywgIm1wcyIsIE5vbmUpIGlzIG5vdCBOb25lIGFuZCB0b3JjaC5iYWNrZW5kcy5tcHMuaXNfYXZhaWxhYmxlKCk6CiAgICAgICAgICAgIF9kZXZpY2UgPSAibXBzIgogICAgICAgIGVsc2U6CiAgICAgICAgICAgIF9kZXZpY2UgPSAiY3B1IgogICAgICAgIGR0eXBlID0gdG9yY2guZmxvYXQxNiBpZiBfZGV2aWNlIGluICgiY3VkYSIsICJtcHMiKSBlbHNlIHRvcmNoLmZsb2F0MzIKICAgICAgICAjIOWEquWFiOaKkyBmcDE2IOiuiumrlO+8iOmrlOepjea4m+WNiuOAgUdQVSDpgannlKjvvInvvJvmspLmnIkgZnAxNiDororpq5TnmoTmqKHlnovlho3pgIDlm57poJDoqK3mrIrph40KICAgICAgICB0cnk6CiAgICAgICAgICAgIHBpcGUgPSBBdXRvUGlwZWxpbmVGb3JUZXh0MkltYWdlLmZyb21fcHJldHJhaW5lZCgKICAgICAgICAgICAgICAgIE1PREVMLCB0b3JjaF9kdHlwZT1kdHlwZSwgdmFyaWFudD0iZnAxNiIsIHVzZV9zYWZldGVuc29ycz1UcnVlCiAgICAgICAgICAgICkKICAgICAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgICAgICBwaXBlID0gQXV0b1BpcGVsaW5lRm9yVGV4dDJJbWFnZS5mcm9tX3ByZXRyYWluZWQoTU9ERUwsIHRvcmNoX2R0eXBlPWR0eXBlKQogICAgICAgIHBpcGUgPSBwaXBlLnRvKF9kZXZpY2UpCiAgICAgICAgcGlwZS5zZXRfcHJvZ3Jlc3NfYmFyX2NvbmZpZyhkaXNhYmxlPVRydWUpCiAgICAgICAgIyBBQUMg55So6YCU77ya6Zec5o6JIHNhZmV0eSBjaGVja2Vy77yM6YG/5YWN6Kqk5Yik5oqK5ZyW5aGX6buRCiAgICAgICAgaWYgaGFzYXR0cihwaXBlLCAic2FmZXR5X2NoZWNrZXIiKToKICAgICAgICAgICAgcGlwZS5zYWZldHlfY2hlY2tlciA9IE5vbmUKICAgICAgICBfcGlwZSA9IHBpcGUKICAgICAgICBwcmludChmIltpbWFnZV0gbW9kZWwgcmVhZHk6IHtNT0RFTH0gb24ge19kZXZpY2V9IiwgZmx1c2g9VHJ1ZSkKICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZXhjOiAgIyBub3FhOiBCTEUwMDEKICAgICAgICBfbG9hZF9lcnIgPSBzdHIoZXhjKQogICAgICAgIHByaW50KGYiW2ltYWdlXSBsb2FkIGZhaWxlZDoge2V4Y30iLCBmbHVzaD1UcnVlKQoKCmRlZiBnZW5lcmF0ZV9wbmcocHJvbXB0OiBzdHIsIHN0ZXBzOiBpbnQpIC0+IGJ5dGVzOgogICAgZ2xvYmFsIF9idXN5LCBfZG9uZSwgX2xhc3RfcHJvbXB0CiAgICBfYnVzeSArPSAxCiAgICBfbGFzdF9wcm9tcHQgPSBwcm9tcHQKICAgIHByaW50KGYiW2ltYWdlXSBnZW5lcmF0aW5nICh7X2J1c3l9IGluIHF1ZXVlKToge3Byb21wdFs6NjBdfSIsIGZsdXNoPVRydWUpCiAgICB0cnk6CiAgICAgICAgd2l0aCBfbG9jazoKICAgICAgICAgICAgaW1hZ2UgPSBfcGlwZSgKICAgICAgICAgICAgICAgIHByb21wdD1wcm9tcHQsCiAgICAgICAgICAgICAgICBudW1faW5mZXJlbmNlX3N0ZXBzPW1heCgxLCBtaW4oc3RlcHMsIDgpKSwKICAgICAgICAgICAgICAgIGd1aWRhbmNlX3NjYWxlPTAuMCwgICAjIHR1cmJvIOezu+WIl+WbuuWumueUqCAwCiAgICAgICAgICAgICkuaW1hZ2VzWzBdCiAgICAgICAgYnVmID0gaW8uQnl0ZXNJTygpCiAgICAgICAgaW1hZ2Uuc2F2ZShidWYsIGZvcm1hdD0iUE5HIikKICAgICAgICBfZG9uZSArPSAxCiAgICAgICAgcHJpbnQoZiJbaW1hZ2VdIGRvbmUgKHRvdGFsIHtfZG9uZX0pOiB7cHJvbXB0Wzo0MF19IiwgZmx1c2g9VHJ1ZSkKICAgICAgICByZXR1cm4gYnVmLmdldHZhbHVlKCkKICAgIGZpbmFsbHk6CiAgICAgICAgX2J1c3kgLT0gMQoKCmNsYXNzIEhhbmRsZXIoQmFzZUhUVFBSZXF1ZXN0SGFuZGxlcik6CiAgICBkZWYgX2pzb24oc2VsZiwgY29kZTogaW50LCBvYmo6IGRpY3QpIC0+IE5vbmU6CiAgICAgICAgYiA9IGpzb24uZHVtcHMob2JqLCBlbnN1cmVfYXNjaWk9RmFsc2UpLmVuY29kZSgidXRmLTgiKQogICAgICAgIHNlbGYuc2VuZF9yZXNwb25zZShjb2RlKQogICAgICAgIHNlbGYuc2VuZF9oZWFkZXIoIkNvbnRlbnQtVHlwZSIsICJhcHBsaWNhdGlvbi9qc29uOyBjaGFyc2V0PXV0Zi04IikKICAgICAgICBzZWxmLnNlbmRfaGVhZGVyKCJDb250ZW50LUxlbmd0aCIsIHN0cihsZW4oYikpKQogICAgICAgIHNlbGYuZW5kX2hlYWRlcnMoKQogICAgICAgIHNlbGYud2ZpbGUud3JpdGUoYikKCiAgICBkZWYgZG9fR0VUKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgaWYgc2VsZi5wYXRoLnN0YXJ0c3dpdGgoIi93aG9hbWkiKToKICAgICAgICAgICAgc2VsZi5fanNvbigyMDAsIHNlbGZfaW5mbygpKQogICAgICAgICAgICByZXR1cm4KICAgICAgICBpZiBzZWxmLnBhdGguc3RhcnRzd2l0aCgiL2JlbmNoIik6CiAgICAgICAgICAgICMg5ZCe5ZCQ6YeP6YeP5ris77ya5Zue5Zu65a6a5aSn5bCP5L2N5YWD57WE6K6T5omL5qmf56uv6KiI5pmC5Lyw566X6YCj57ea6YCf5bqm77yI5omL5qmfIE9mZmxvYWRNYW5hZ2VyIOeUqO+8ieOAggogICAgICAgICAgICBuID0gMjAwXzAwMAogICAgICAgICAgICBpZiAiYnl0ZXM9IiBpbiBzZWxmLnBhdGg6CiAgICAgICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICAgICAgbiA9IG1heCgxLCBtaW4oaW50KHNlbGYucGF0aC5zcGxpdCgiYnl0ZXM9IilbMV0uc3BsaXQoIiYiKVswXSksIDVfMDAwXzAwMCkpCiAgICAgICAgICAgICAgICBleGNlcHQgRXhjZXB0aW9uOiAgIyBub3FhOiBCTEUwMDEKICAgICAgICAgICAgICAgICAgICBuID0gMjAwXzAwMAogICAgICAgICAgICBzZWxmLnNlbmRfcmVzcG9uc2UoMjAwKQogICAgICAgICAgICBzZWxmLnNlbmRfaGVhZGVyKCJDb250ZW50LVR5cGUiLCAiYXBwbGljYXRpb24vb2N0ZXQtc3RyZWFtIikKICAgICAgICAgICAgc2VsZi5zZW5kX2hlYWRlcigiQ29udGVudC1MZW5ndGgiLCBzdHIobikpCiAgICAgICAgICAgIHNlbGYuZW5kX2hlYWRlcnMoKQogICAgICAgICAgICBzZWxmLndmaWxlLndyaXRlKGIiXDAiICogbikKICAgICAgICAgICAgcmV0dXJuCiAgICAgICAgaWYgc2VsZi5wYXRoLnN0YXJ0c3dpdGgoIi9oZWFsdGgiKToKICAgICAgICAgICAgc2VsZi5fanNvbigyMDAsIHsKICAgICAgICAgICAgICAgICJvayI6IFRydWUsCiAgICAgICAgICAgICAgICAicmVhZHkiOiBfcGlwZSBpcyBub3QgTm9uZSwKICAgICAgICAgICAgICAgICJtb2RlbCI6IE1PREVMLAogICAgICAgICAgICAgICAgImRldmljZSI6IF9kZXZpY2UsCiAgICAgICAgICAgICAgICAiZXJyb3IiOiBfbG9hZF9lcnIsCiAgICAgICAgICAgICAgICAiYnVzeSI6IF9idXN5LAogICAgICAgICAgICAgICAgImdlbmVyYXRpbmciOiBfYnVzeSA+IDAsCiAgICAgICAgICAgICAgICAiZG9uZSI6IF9kb25lLAogICAgICAgICAgICAgICAgImxhc3RfcHJvbXB0IjogX2xhc3RfcHJvbXB0LAogICAgICAgICAgICB9KQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHNlbGYuX2pzb24oNDA0LCB7Im9rIjogRmFsc2V9KQoKICAgIGRlZiBkb19QT1NUKHNlbGYpIC0+IE5vbmU6CiAgICAgICAgaWYgbm90IHNlbGYucGF0aC5zdGFydHN3aXRoKCIvZ2VuZXJhdGUiKToKICAgICAgICAgICAgc2VsZi5fanNvbig0MDQsIHsib2siOiBGYWxzZX0pCiAgICAgICAgICAgIHJldHVybgogICAgICAgIGlmIF9waXBlIGlzIE5vbmU6CiAgICAgICAgICAgIHNlbGYuX2pzb24oNTAzLCB7Im9rIjogRmFsc2UsICJlcnJvciI6IF9sb2FkX2VyciBvciAi5qih5Z6L5bCa5pyq6LyJ5YWl5a6M5oiQ77yM6KuL56iN5YCZ5YaN6KmmIn0pCiAgICAgICAgICAgIHJldHVybgogICAgICAgIHRyeToKICAgICAgICAgICAgbiA9IGludChzZWxmLmhlYWRlcnMuZ2V0KCJDb250ZW50LUxlbmd0aCIsICIwIikgb3IgIjAiKQogICAgICAgICAgICBib2R5ID0ganNvbi5sb2FkcyhzZWxmLnJmaWxlLnJlYWQobikgb3IgYiJ7fSIpCiAgICAgICAgICAgIHByb21wdCA9IChib2R5LmdldCgicHJvbXB0Iikgb3IgIiIpLnN0cmlwKCkKICAgICAgICAgICAgc3RlcHMgPSBpbnQoYm9keS5nZXQoInN0ZXBzIikgb3IgREVGQVVMVF9TVEVQUykKICAgICAgICAgICAgaWYgbm90IHByb21wdDoKICAgICAgICAgICAgICAgIHNlbGYuX2pzb24oNDAwLCB7Im9rIjogRmFsc2UsICJlcnJvciI6ICLnvLrlsJEgcHJvbXB0In0pCiAgICAgICAgICAgICAgICByZXR1cm4KICAgICAgICAgICAgcG5nID0gZ2VuZXJhdGVfcG5nKHByb21wdCwgc3RlcHMpCiAgICAgICAgICAgIHNlbGYuc2VuZF9yZXNwb25zZSgyMDApCiAgICAgICAgICAgIHNlbGYuc2VuZF9oZWFkZXIoIkNvbnRlbnQtVHlwZSIsICJpbWFnZS9wbmciKQogICAgICAgICAgICBzZWxmLnNlbmRfaGVhZGVyKCJDb250ZW50LUxlbmd0aCIsIHN0cihsZW4ocG5nKSkpCiAgICAgICAgICAgIHNlbGYuZW5kX2hlYWRlcnMoKQogICAgICAgICAgICBzZWxmLndmaWxlLndyaXRlKHBuZykKICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGV4YzogICMgbm9xYTogQkxFMDAxCiAgICAgICAgICAgIHNlbGYuX2pzb24oNTAwLCB7Im9rIjogRmFsc2UsICJlcnJvciI6IHN0cihleGMpfSkKCiAgICBkZWYgbG9nX21lc3NhZ2Uoc2VsZiwgZm10OiBzdHIsICphcmdzKSAtPiBOb25lOgogICAgICAgIHN5cy5zdGRlcnIud3JpdGUoIlslc10gJXNcbiIgJSAodGltZS5zdHJmdGltZSgiJUg6JU06JVMiKSwgZm10ICUgYXJncykpCgoKZGVmIG1haW4oKSAtPiBOb25lOgogICAgdGhyZWFkaW5nLlRocmVhZCh0YXJnZXQ9bG9hZF9waXBlLCBkYWVtb249VHJ1ZSkuc3RhcnQoKQogICAgIyDpoJDoqK0gMC4wLjAuMO+8iOaJi+apnyBXaS1GaSDpnIDopoHvvInvvJvlj6rnlKggVVNCKGFkYiByZXZlcnNlKSDnmoTkurrlj6/oqK0gU01MX0JJTkQ9MTI3LjAuMC4xIOmOluacrOapn+OAggogICAgYmluZCA9IG9zLmVudmlyb24uZ2V0KCJTTUxfQklORCIsICIwLjAuMC4wIikKICAgIHNlcnZlciA9IFRocmVhZGluZ0hUVFBTZXJ2ZXIoKGJpbmQsIFBPUlQpLCBIYW5kbGVyKQogICAgcHJpbnQoZiJTcGVhayBNeSBMaWZlIGltYWdlIHNlcnZlcjogaHR0cDovL3tiaW5kfTp7UE9SVH0gKG1vZGVsIHtNT0RFTH0pIiwgZmx1c2g9VHJ1ZSkKICAgIHNlcnZlci5zZXJ2ZV9mb3JldmVyKCkKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg=='))
    procs['image']=subprocess.Popen(['python','/content/img.py'],env=dict(os.environ,SML_IMAGE_PORT='9881',SML_BIND='127.0.0.1'),
        stdout=open('/content/img.log','w'),stderr=subprocess.STDOUT)
    _wait('http://127.0.0.1:9881/health','生圖 SD-Turbo')
if RUN_TEXT:
    open('/content/txt.py','wb').write(base64.b64decode('IyEvdXNyL2Jpbi9lbnYgcHl0aG9uMwoiIiLmi77oqp7ogIUgU3BlYWsgTXkgTGlmZSDmnKzmqZ/mloflrZfmjqjoq5bkvLrmnI3lmajvvIhNYWMg56uv77yM5YWN6YeR6ZGw77yJ44CCCgrnlbbmiYvmqZ/oiIfpm7vohabpgKPnt5rlpKDlv6vvvIjmnInnt5ogVVNCIOaIlumrmOmAnyBXaS1Gae+8ieaZgu+8jOaLvuiqnuiAheacg+aKiuOAjOWPpeWtkOmHjee1hOOAjemAmemghQrmloflrZfpgYvnrpfkuZ/kuqTntabpm7vohabvvIh0aWVyLUZVTEzvvInvvIznm6Hph4/muJvlsJHmiYvmqZ/osqDmk5TjgILpgJnmlK/kvLrmnI3lmajlsLHmmK/pm7vohabnq6/nmoTmloflrZflpKfohabvvJoK55SoIHRyYW5zZm9ybWVycyDlnKggQXBwbGUgTVBTIOS4iui3keS4gOWAi+Wwj+WeiyBpbnN0cnVjdCDmqKHlnovvvIjpoJDoqK0gUXdlbjIuNS0xLjVCLUluc3RydWN077yMCuWkmuiqnu+8muS4re+8j+iLse+8j+aXpe+8j+mfk++8ie+8jOmmluasoeWft+ihjOacg+iHquWLleS4i+i8ie+8iOWFjemHkemRsO+8ieOAggoK56uv6bue77yaCiAgICBHRVQgIC9oZWFsdGggICAgICAgIC0+IHtvaywgcmVhZHksIG1vZGVsLCBkZXZpY2UsIGVycm9yfQogICAgUE9TVCAvcmVjb25zdHJ1Y3QgICBib2R5IHsic3lzdGVtIjogIi4uLiIsICJ1c2VyIjogIi4uLiJ9IC0+IHsidGV4dCI6ICIuLi4ifQoK55Kw5aKD6K6K5pW477yaCiAgICBURVhUX0lORkVSX01PREVM77yI6aCQ6KitIFF3ZW4vUXdlbjIuNS0xLjVCLUluc3RydWN077yb5oOz5pu05b+r5Y+v55SoIFF3ZW4vUXdlbjIuNS0wLjVCLUluc3RydWN077yJCiAgICBTTUxfVEVYVF9QT1JU77yI6aCQ6KitIDk4ODLvvIkKIiIiCmZyb20gX19mdXR1cmVfXyBpbXBvcnQgYW5ub3RhdGlvbnMKCmltcG9ydCBqc29uCmltcG9ydCBvcwppbXBvcnQgc3lzCmltcG9ydCB0aHJlYWRpbmcKaW1wb3J0IHRpbWUKZnJvbSBodHRwLnNlcnZlciBpbXBvcnQgQmFzZUhUVFBSZXF1ZXN0SGFuZGxlciwgVGhyZWFkaW5nSFRUUFNlcnZlcgoKUE9SVCA9IGludChvcy5lbnZpcm9uLmdldCgiU01MX1RFWFRfUE9SVCIsICI5ODgyIikpCk1PREVMID0gb3MuZW52aXJvbi5nZXQoIlRFWFRfSU5GRVJfTU9ERUwiLCAiUXdlbi9Rd2VuMi41LTEuNUItSW5zdHJ1Y3QiKQpNQVhfTkVXX1RPS0VOUyA9IGludChvcy5lbnZpcm9uLmdldCgiVEVYVF9JTkZFUl9NQVhfVE9LRU5TIiwgIjk2IikpCgpfbW9kZWwgPSBOb25lCl90b2sgPSBOb25lCl9sb2NrID0gdGhyZWFkaW5nLkxvY2soKSAgICAgICAjIOW6j+WIl+WMluaOqOirlu+8jOmBv+WFjSBNUFMg6KiY5oa26auU54iG5o6JCl9sb2FkX2VyciA9ICIiCl9kZXZpY2UgPSAiY3B1IgoKCmRlZiBsb2FkX21vZGVsKCkgLT4gTm9uZToKICAgICIiIuiDjOaZr+i8ieWFpeaooeWei++8iOmmluasoeacg+iHquWLleS4i+i8ie+8ie+8jOiukyBzZXJ2ZXIg6IO956uL5Yi75Zue5oeJIC9oZWFsdGjjgIIiIiIKICAgIGdsb2JhbCBfbW9kZWwsIF90b2ssIF9sb2FkX2VyciwgX2RldmljZQogICAgdHJ5OgogICAgICAgIGltcG9ydCB0b3JjaAogICAgICAgIGZyb20gdHJhbnNmb3JtZXJzIGltcG9ydCBBdXRvTW9kZWxGb3JDYXVzYWxMTSwgQXV0b1Rva2VuaXplcgoKICAgICAgICAjIOijnee9ruWEquWFiOW6j++8mkNVREHvvIhXaW5kb3dzL0xpbnV4IE5WSURJQe+8ieKGkiBNUFPvvIhBcHBsZe+8ieKGkiBDUFXjgIIKICAgICAgICBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpOgogICAgICAgICAgICBfZGV2aWNlID0gImN1ZGEiCiAgICAgICAgZWxpZiBnZXRhdHRyKHRvcmNoLmJhY2tlbmRzLCAibXBzIiwgTm9uZSkgaXMgbm90IE5vbmUgYW5kIHRvcmNoLmJhY2tlbmRzLm1wcy5pc19hdmFpbGFibGUoKToKICAgICAgICAgICAgX2RldmljZSA9ICJtcHMiCiAgICAgICAgZWxzZToKICAgICAgICAgICAgX2RldmljZSA9ICJjcHUiCiAgICAgICAgZHR5cGUgPSB0b3JjaC5mbG9hdDE2IGlmIF9kZXZpY2UgaW4gKCJjdWRhIiwgIm1wcyIpIGVsc2UgdG9yY2guZmxvYXQzMgogICAgICAgIF90b2sgPSBBdXRvVG9rZW5pemVyLmZyb21fcHJldHJhaW5lZChNT0RFTCkKICAgICAgICBtb2RlbCA9IEF1dG9Nb2RlbEZvckNhdXNhbExNLmZyb21fcHJldHJhaW5lZChNT0RFTCwgdG9yY2hfZHR5cGU9ZHR5cGUpCiAgICAgICAgbW9kZWwgPSBtb2RlbC50byhfZGV2aWNlKQogICAgICAgIG1vZGVsLmV2YWwoKQogICAgICAgIF9tb2RlbCA9IG1vZGVsCiAgICAgICAgcHJpbnQoZiJbdGV4dF0gbW9kZWwgcmVhZHk6IHtNT0RFTH0gb24ge19kZXZpY2V9IiwgZmx1c2g9VHJ1ZSkKICAgIGV4Y2VwdCBFeGNlcHRpb24gYXMgZXhjOiAgIyBub3FhOiBCTEUwMDEKICAgICAgICBfbG9hZF9lcnIgPSBzdHIoZXhjKQogICAgICAgIHByaW50KGYiW3RleHRdIGxvYWQgZmFpbGVkOiB7ZXhjfSIsIGZsdXNoPVRydWUpCgoKZGVmIHJlY29uc3RydWN0KHN5c3RlbTogc3RyLCB1c2VyOiBzdHIsIHRlbXBlcmF0dXJlOiBmbG9hdCA9IDAuNSkgLT4gc3RyOgogICAgaW1wb3J0IHRvcmNoCgogICAgIyDoqZXliIbpoZ7ku7vli5nmnIPlgrPkvY7muqvvvIgwLjF+MC4y77yJ5rGC56mp5a6a77yb5aS+5Zyo5a6J5YWo56+E5ZyNCiAgICB0ZW1wZXJhdHVyZSA9IG1heCgwLjA1LCBtaW4oMS41LCBmbG9hdCh0ZW1wZXJhdHVyZSBvciAwLjUpKSkKICAgIG1lc3NhZ2VzID0gW10KICAgIGlmIHN5c3RlbS5zdHJpcCgpOgogICAgICAgIG1lc3NhZ2VzLmFwcGVuZCh7InJvbGUiOiAic3lzdGVtIiwgImNvbnRlbnQiOiBzeXN0ZW19KQogICAgbWVzc2FnZXMuYXBwZW5kKHsicm9sZSI6ICJ1c2VyIiwgImNvbnRlbnQiOiB1c2VyfSkKICAgIHdpdGggX2xvY2s6CiAgICAgICAgcHJvbXB0ID0gX3Rvay5hcHBseV9jaGF0X3RlbXBsYXRlKAogICAgICAgICAgICBtZXNzYWdlcywgdG9rZW5pemU9RmFsc2UsIGFkZF9nZW5lcmF0aW9uX3Byb21wdD1UcnVlCiAgICAgICAgKQogICAgICAgIGlucHV0cyA9IF90b2socHJvbXB0LCByZXR1cm5fdGVuc29ycz0icHQiKS50byhfZGV2aWNlKQogICAgICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOgogICAgICAgICAgICBvdXQgPSBfbW9kZWwuZ2VuZXJhdGUoCiAgICAgICAgICAgICAgICAqKmlucHV0cywKICAgICAgICAgICAgICAgIG1heF9uZXdfdG9rZW5zPU1BWF9ORVdfVE9LRU5TLAogICAgICAgICAgICAgICAgZG9fc2FtcGxlPVRydWUsCiAgICAgICAgICAgICAgICB0ZW1wZXJhdHVyZT10ZW1wZXJhdHVyZSwKICAgICAgICAgICAgICAgIHRvcF9wPTAuOSwKICAgICAgICAgICAgICAgIHBhZF90b2tlbl9pZD1fdG9rLmVvc190b2tlbl9pZCwKICAgICAgICAgICAgKQogICAgICAgIGdlbiA9IG91dFswXVtpbnB1dHNbImlucHV0X2lkcyJdLnNoYXBlWzFdOl0KICAgICAgICB0ZXh0ID0gX3Rvay5kZWNvZGUoZ2VuLCBza2lwX3NwZWNpYWxfdG9rZW5zPVRydWUpLnN0cmlwKCkKICAgIHRleHQgPSAodGV4dCBvciAiIikuc3RyaXAoKQogICAgIyBKU09OIOi8uOWHuu+8iOW+qeWBpeipleWIhuetiee1kOani+WMluS7u+WLme+8ieW/hemgiOWujOaVtOS/neeVmeKAlOKAlOWkmuihjCBKU09OIOegjeWIsOesrOS4gOihjOacg+aVtOWMheWjnuaOie+8jAogICAgIyDlrqLmiLbnq6/op6PmnpDlpLHmlZflsLHmgoTmgoTmjonliLDliKXnmoToqZXliIbkvobmupDvvIzliIbmlbjlv73pq5jlv73kvY7jgIIKICAgICMg5qih5Z6L5bi45YyFIGBgYGpzb24g5ZyN5qyE5oiW5Yqg5YmN6KiA77yM5omA5Lul5LiN6IO95Y+q55yL6ZaL6aCt77ya5Y+W56ys5LiA5YCLIHsg5Yiw5pyA5b6M5LiA5YCLIH3vvIzog70ganNvbiDop6PmnpDmiY3nrpfjgIIKICAgIGlmICJ7IiBpbiB0ZXh0OgogICAgICAgIHNpLCBlaSA9IHRleHQuZmluZCgieyIpLCB0ZXh0LnJmaW5kKCJ9IikKICAgICAgICBpZiBlaSA+IHNpOgogICAgICAgICAgICBjYW5kID0gdGV4dFtzaTplaSArIDFdCiAgICAgICAgICAgIHRyeToKICAgICAgICAgICAgICAgIGpzb24ubG9hZHMoY2FuZCkKICAgICAgICAgICAgICAgIHJldHVybiBjYW5kCiAgICAgICAgICAgIGV4Y2VwdCBFeGNlcHRpb246CiAgICAgICAgICAgICAgICBwYXNzCiAgICAjIOWFtumkmO+8iEFBQyDntYTlj6XnrYnjgIzkuIDlj6XoqbHjgI3pnIDmsYLvvInntq3mjIHlj6rlj5bnrKzkuIDooYzjgIHljrvlvJXomZ8KICAgIHRleHQgPSB0ZXh0LnNwbGl0bGluZXMoKVswXS5zdHJpcCgpIGlmIHRleHQgZWxzZSAiIgogICAgcmV0dXJuIHRleHQuc3RyaXAoKS5zdHJpcCgi44CM44CNXCInIikKCgpjbGFzcyBIYW5kbGVyKEJhc2VIVFRQUmVxdWVzdEhhbmRsZXIpOgogICAgZGVmIF9qc29uKHNlbGYsIGNvZGU6IGludCwgb2JqOiBkaWN0KSAtPiBOb25lOgogICAgICAgIGIgPSBqc29uLmR1bXBzKG9iaiwgZW5zdXJlX2FzY2lpPUZhbHNlKS5lbmNvZGUoInV0Zi04IikKICAgICAgICBzZWxmLnNlbmRfcmVzcG9uc2UoY29kZSkKICAgICAgICBzZWxmLnNlbmRfaGVhZGVyKCJDb250ZW50LVR5cGUiLCAiYXBwbGljYXRpb24vanNvbjsgY2hhcnNldD11dGYtOCIpCiAgICAgICAgc2VsZi5zZW5kX2hlYWRlcigiQ29udGVudC1MZW5ndGgiLCBzdHIobGVuKGIpKSkKICAgICAgICBzZWxmLmVuZF9oZWFkZXJzKCkKICAgICAgICBzZWxmLndmaWxlLndyaXRlKGIpCgogICAgZGVmIGRvX0dFVChzZWxmKSAtPiBOb25lOgogICAgICAgIGlmIHNlbGYucGF0aC5zdGFydHN3aXRoKCIvaGVhbHRoIik6CiAgICAgICAgICAgIHNlbGYuX2pzb24oMjAwLCB7CiAgICAgICAgICAgICAgICAib2siOiBUcnVlLAogICAgICAgICAgICAgICAgInJlYWR5IjogX21vZGVsIGlzIG5vdCBOb25lLAogICAgICAgICAgICAgICAgIm1vZGVsIjogTU9ERUwsCiAgICAgICAgICAgICAgICAiZGV2aWNlIjogX2RldmljZSwKICAgICAgICAgICAgICAgICJlcnJvciI6IF9sb2FkX2VyciwKICAgICAgICAgICAgfSkKICAgICAgICBlbHNlOgogICAgICAgICAgICBzZWxmLl9qc29uKDQwNCwgeyJvayI6IEZhbHNlfSkKCiAgICBkZWYgZG9fUE9TVChzZWxmKSAtPiBOb25lOgogICAgICAgIGlmIG5vdCBzZWxmLnBhdGguc3RhcnRzd2l0aCgiL3JlY29uc3RydWN0Iik6CiAgICAgICAgICAgIHNlbGYuX2pzb24oNDA0LCB7Im9rIjogRmFsc2V9KQogICAgICAgICAgICByZXR1cm4KICAgICAgICBpZiBfbW9kZWwgaXMgTm9uZToKICAgICAgICAgICAgc2VsZi5fanNvbig1MDMsIHsib2siOiBGYWxzZSwgImVycm9yIjogX2xvYWRfZXJyIG9yICLmqKHlnovlsJrmnKrovInlhaXlrozmiJDvvIzoq4vnqI3lgJnlho3oqaYifSkKICAgICAgICAgICAgcmV0dXJuCiAgICAgICAgdHJ5OgogICAgICAgICAgICBuID0gaW50KHNlbGYuaGVhZGVycy5nZXQoIkNvbnRlbnQtTGVuZ3RoIiwgIjAiKSBvciAiMCIpCiAgICAgICAgICAgIGJvZHkgPSBqc29uLmxvYWRzKHNlbGYucmZpbGUucmVhZChuKSBvciBiInt9IikKICAgICAgICAgICAgc3lzdGVtID0gKGJvZHkuZ2V0KCJzeXN0ZW0iKSBvciAiIikuc3RyaXAoKQogICAgICAgICAgICB1c2VyID0gKGJvZHkuZ2V0KCJ1c2VyIikgb3IgIiIpLnN0cmlwKCkKICAgICAgICAgICAgaWYgbm90IHVzZXI6CiAgICAgICAgICAgICAgICBzZWxmLl9qc29uKDQwMCwgeyJvayI6IEZhbHNlLCAiZXJyb3IiOiAi57y65bCRIHVzZXIifSkKICAgICAgICAgICAgICAgIHJldHVybgogICAgICAgICAgICB0ZW1wZXJhdHVyZSA9IGJvZHkuZ2V0KCJ0ZW1wZXJhdHVyZSIsIDAuNSkKICAgICAgICAgICAgdGV4dCA9IHJlY29uc3RydWN0KHN5c3RlbSwgdXNlciwgdGVtcGVyYXR1cmUpCiAgICAgICAgICAgIHNlbGYuX2pzb24oMjAwLCB7Im9rIjogVHJ1ZSwgInRleHQiOiB0ZXh0fSkKICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGV4YzogICMgbm9xYTogQkxFMDAxCiAgICAgICAgICAgIHNlbGYuX2pzb24oNTAwLCB7Im9rIjogRmFsc2UsICJlcnJvciI6IHN0cihleGMpfSkKCiAgICBkZWYgbG9nX21lc3NhZ2Uoc2VsZiwgZm10OiBzdHIsICphcmdzKSAtPiBOb25lOgogICAgICAgIHN5cy5zdGRlcnIud3JpdGUoIlslc10gJXNcbiIgJSAodGltZS5zdHJmdGltZSgiJUg6JU06JVMiKSwgZm10ICUgYXJncykpCgoKZGVmIG1haW4oKSAtPiBOb25lOgogICAgdGhyZWFkaW5nLlRocmVhZCh0YXJnZXQ9bG9hZF9tb2RlbCwgZGFlbW9uPVRydWUpLnN0YXJ0KCkKICAgICMg6aCQ6KitIDAuMC4wLjDvvIjmiYvmqZ8gV2ktRmkg6ZyA6KaB77yJ77yb5Y+q55SoIFVTQihhZGIgcmV2ZXJzZSkg55qE5Lq65Y+v6KitIFNNTF9CSU5EPTEyNy4wLjAuMSDpjpbmnKzmqZ/jgIIKICAgIGJpbmQgPSBvcy5lbnZpcm9uLmdldCgiU01MX0JJTkQiLCAiMC4wLjAuMCIpCiAgICBzZXJ2ZXIgPSBUaHJlYWRpbmdIVFRQU2VydmVyKChiaW5kLCBQT1JUKSwgSGFuZGxlcikKICAgIHByaW50KGYiU3BlYWsgTXkgTGlmZSB0ZXh0IHNlcnZlcjogaHR0cDovL3tiaW5kfTp7UE9SVH0gKG1vZGVsIHtNT0RFTH0pIiwgZmx1c2g9VHJ1ZSkKICAgIHNlcnZlci5zZXJ2ZV9mb3JldmVyKCkKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg=='))
    procs['text']=subprocess.Popen(['python','/content/txt.py'],env=dict(os.environ,SML_TEXT_PORT='9882',SML_BIND='127.0.0.1'),
        stdout=open('/content/txt.log','w'),stderr=subprocess.STDOUT)
    _wait('http://127.0.0.1:9882/health','文字 Qwen')
print('服務啟動完成')


## ⑥ 啟動雲端橋接（:9879，app + web 共用，含 CORS、動態偵測三功能）


In [ ]:
open('/content/bridge.py','wb').write(base64.b64decode('...'))
env=dict(os.environ, GSV_ROOT=ROOT, MODELS_ROOT=MODELS_ROOT, HISTORY_PATH=f'{ROOT}/user_models/model_history.json',
         API_BASE='http://127.0.0.1:9880', IMAGE_BASE='http://127.0.0.1:9881', TEXT_BASE='http://127.0.0.1:9882',
         BRIDGE_PORT='9879')
procs['bridge']=subprocess.Popen(['python','/content/bridge.py'],env=env,
    stdout=open('/content/bridge.log','w'),stderr=subprocess.STDOUT)
time.sleep(3)
with urllib.request.urlopen('http://127.0.0.1:9879/health',timeout=5) as r:
    print('✅ 橋接就緒：', r.read().decode())


## ⑦ 開公開網址（ngrok，授權碼從雲端自動取用）

授權碼**存在你的拾語者帳號雲端**（在 App／網頁的「ngrok 雲端通道」填一次即可），
Colab 只需要一組**非機密的配對碼**就能自動取回授權碼——你不用把授權碼貼進 Colab。

1. App／網頁 → 設定 → **ngrok 雲端通道** → 填 authtoken（＋建議填固定網域）→ 複製「配對碼」。
2. Colab 左側 **🔑 Secrets → 新增** 名稱 `SML_PAIR_CODE`，值貼上那組配對碼（更名前設的 `VW_PAIR_CODE` 也照樣讀得到）（打開 Notebook access）。
   （或不設 Secret，執行本格時會跳出輸入框讓你貼配對碼。）

> 之後每次重開 Colab 都自動從雲端拿授權碼；換 ngrok 授權碼只要在 App／網頁改，Colab 不用動。


In [ ]:
!pip -q install pyngrok
from pyngrok import ngrok, conf
import getpass, json, urllib.request

FIREBASE_PROJECT = 'voiceweaver-af4da'   # 公開的專案 ID（Firebase 專案 ID 建立後不能改名）

def _secret(name):
    try:
        from google.colab import userdata; return (userdata.get(name) or '').strip()
    except Exception:
        return ''

# 配對碼（非機密）：優先讀 Colab Secret『SML_PAIR_CODE』（更名前的『VW_PAIR_CODE』也認），否則執行時輸入。
pair = _secret('SML_PAIR_CODE') or _secret('VW_PAIR_CODE') or input('貼上 App／網頁的「配對碼」：').strip()
assert pair, '需要配對碼'

# 用配對碼從 Firestore 取回你在 App／網頁填的 ngrok 授權碼（該文件只有拿到配對碼才讀得到）。
url = f'https://firestore.googleapis.com/v1/projects/{FIREBASE_PROJECT}/databases/(default)/documents/cloudbridge/{pair}'
with urllib.request.urlopen(url, timeout=15) as r:
    fields = json.load(r).get('fields', {})
token  = fields.get('ngrokToken', {}).get('stringValue', '').strip()
domain = fields.get('ngrokDomain', {}).get('stringValue', '').strip()
# 清洗網域：使用者常把整串網址貼進來（https://xxx.ngrok-free.app/）——
# ngrok 的 domain 參數只吃純網域，帶 scheme/斜線會 ERR_NGROK_9038 開不了通道。
import re as _re
domain = _re.sub(r'^https?://', '', domain).split('/')[0].strip().rstrip('.')
assert token, '雲端沒有 ngrok 授權碼——請先在 App／網頁的「ngrok 雲端通道」填入並確認已同步'
print('✅ 已從雲端取回 ngrok 授權碼（未顯示內容）', '｜固定網域：'+domain if domain else '｜無固定網域')

conf.get_default().auth_token = token
ngrok.kill()
opts = {'addr': 9879, 'proto': 'http'}
if domain: opts['domain'] = domain
pub = ngrok.connect(**opts).public_url
print('\n'+'='*54+f'\n  公開網址：{pub}\n'+'='*54)
print('\n📱 App：設定 → GPT-SoVITS API 位址 →「新增」貼上：\n   ', pub)
print('\n🌐 網頁：設定 → 讓電腦幫忙跑運算 →「新增雲端」貼上：\n   ', pub)
print('\n（固定網域下此網址永不變，App／網頁填一次即可）' if domain else '\n（未設固定網域：網址每次會變，建議在 App／網頁填固定網域）')


## 使用說明

**ngrok 授權碼完全不進 Colab**：在 App／網頁的「ngrok 雲端通道」填一次 → 存你的帳號雲端 → Colab 用配對碼自動取回。

| 客戶端 | 填哪裡 |
|---|---|
| 📱 Android App | 設定 → GPT-SoVITS API 位址 → 新增 → 貼 ngrok 網址（可多台、失敗接手）|
| 🌐 網頁版 | 設定 → 讓電腦幫忙跑運算 → 新增雲端 → 貼 ngrok 網址 |
| ☁️ ngrok 授權碼 | App／網頁 → 設定 → ngrok 雲端通道（存雲端，跨裝置同步）|

- **自由選擇功能**：回儲存格 ① 改 `RUN_VOICE/IMAGE/TEXT` 再重跑。
- **自由新增多台**：App 與網頁都可加多個網址，連不上自動換下一個。
- 設了 ngrok **固定網域**，App／網頁填一次網址永不變；否則每次重開要重貼。
- Colab 免費版閒置約 90 分鐘、單次最長約 12 小時回收；回收後重跑 ⑤⑥⑦。
